In [53]:
import numpy as np

In [54]:
def building_heat_loss(T_internal_F, T_external_F, R_wall, area, ACH, volume):
    """
    Calculates total heat loss rate from building per second
    in Watts, given internal and external temperature

    T_internal_F: Temperature inside the room in Fahrenheit
    T_external_F: Temperature outside the building in Fahrenheit
    R_wall: insulation R value of wall
    ACH: Air Changes per Hour by infiltration
    volume:
    area: total surface area through which heat can escape (walls, ceiling, floor, windows)
    """
    T_internal_K = (T_internal_F - 32) * 5/9 + 273.15
    T_external_K = (T_external_F - 32) * 5/9 + 273.15

    delta_T = T_internal_K - T_external_K
    # conductive heat loss through building
    Q_conductive = (delta_T / R_wall) * area

    # infiltration loss - air leaking in and out
    rho_air = 1.2 # unit - kg/m3
    cp_air = 1005 # unit J/kg.K
    Q_infiltration = (ACH / 3600) * volume * rho_air * cp_air * delta_T

    Q_total = Q_conductive + Q_infiltration

    #print("Q conductive: ", Q_conductive)
    #print("Q infiltration: ", Q_infiltration)
    return Q_total
    # return Q_conductive

In [55]:
def sat_pressure(T_F):
    """
    Calculates saturation pressure, the maximum pressure 
    water vapor can exert at a given temperature
    
    T_F: Temperature in Fahrenheit
    """
    # Convert to Celsius for ASHRAE Eq. 5 (IAPWS-IF97)
    # ASHRAE Fundamentals 2025, Chapter 1, Equation 5
    T_C = (T_F - 32) * 5/9
    T_K = T_C + 273.15
    theta = T_K + (-0.238555575678e0) / (T_K - 0.650175348448e3)
    A = theta**2 + 0.116705214528e4 * theta - 0.724213167032e6
    B = -0.170738469401e2 * theta**2 + 0.120208247025e5 * theta - 0.323255503223e7
    C = 0.149151086135e2 * theta**2 - 0.482326573616e4 * theta + 0.405113405421e6
    p_ws = 1000 * (2*C / (-B + (B**2 - 4*A*C)**0.5))**4  # kPa
    return p_ws

In [56]:
def indoor_humidity(T_internal_F, T_external_F, RH_external):
    """
    Calculates indoor relative humidity by assuming indoor air 
    carries the same absolute moisture content as outdoor air, 
    then recalculates what that moisture level feels like at 
    the warmer indoor temperature

    T_internal_F: Indoor temperature in Fahrenheit
    T_external_F: External Temperature in Fahrenheit
    RH_external: External Relative Humidity
    """
    """
    ASHRAE Fundamentals 2025 Chapter 1
    Humidity ratio W: Equation 21
    Saturation pressure: Equation 5
    """
    P_atm = 101.325  # kPa

    # Outdoor absolute humidity (humidity ratio W)
    P_sat_ext = sat_pressure(T_external_F)
    p_w = RH_external * P_sat_ext
    W = 0.621945 * p_w / (P_atm - p_w)  # Eq. 21 ASHRAE Ch.1

    # Indoor RH at indoor temp, same W (infiltration assumption)
    P_sat_int = sat_pressure(T_internal_F)
    p_w_int = W * P_atm / (0.621945 + W)
    RH_internal = p_w_int / P_sat_int
    RH_internal = min(RH_internal, 1.0)

    return RH_internal

In [57]:
def wall_temp(T_internal_F, Q_total, area, R_film=0.13):
    """
    Calculates inner wall temperature at steady state

    T_internal_F: Temperature inside the room in Fahrenheit
    R_film: insulation R value of the layer of air next to the inner faces of the wall
    area: total surface area through which heat can escape (walls, ceiling, floor, windows)
    """
    T_internal_K = (T_internal_F - 32) * 5/9 + 273.15    
    T_wall_K = T_internal_K - (Q_total * R_film / area) 
    T_wall_F = (T_wall_K - 273.15) * 9/5 + 32  
    return T_wall_F

In [58]:
def comfort_model(T_internal_F, RH_internal, T_wall_F, clothing_factor=0.5):
    """
    Calculates total heat loss from the human body via radiation to walls, 
    convection to air, and evaporation. Compares against metabolic rate to 
    give a Hot/Cold/Good verdict.

    T_internal_F: Indoor Temperature in Fahrenheit
    RH_internal: Indoor Relative Humidity
    T_wall_F: Wall Temperature in Fahrenheit
    """
    
    """
    Radiation + convection heat loss from human body
    Taken from Dr. Gray's matlab code
    ASHRAE Fundamentals 2025 Chapter 9
    """
    
    epsilon = 0.98
    sigma = 5.6703e-8
    H = 3.5 # Lowered convection coefficient from 5 to 3.5
    T_skin_K = 306.15 # 33 C

    T_wall_K = (T_wall_F - 32) * 5/9 + 273.15
    T_air_K  = (T_internal_F - 32) * 5/9 + 273.15

    q_rad  = epsilon * sigma * (T_skin_K**4 - T_wall_K**4)
    q_conv = H * (T_skin_K - T_air_K)

    q_rad *= clothing_factor
    q_conv *= clothing_factor
    
    # Evaporative heat loss — ASHRAE Fundamentals Ch.9
    # q_evap decreases as RH increases (less evaporation possible)
    # At rest: ~10 W/m2 at low RH, approaches 0 at high RH
    # Tweaked to 5
    q_evap = 5 * (1 - RH_internal)  # simplified linear approximation
    q_total = q_rad + q_conv + q_evap

    METABOLIC_RATE = 60  # W/m2, seated quiet, ASHRAE Fundamentals Ch.9 Table 4

    tolerance = 20  # W/m2 either side — needs calibration from your survey data

    if q_total < METABOLIC_RATE - tolerance:
        verdict = "Hot"
    elif q_total > METABOLIC_RATE + tolerance:
        verdict = "Cold"
    else:
        verdict = "Good"

    return q_total, verdict




In [70]:
def steady_state_model(T_setpoint_F, T_external_F, RH_external, R_wall, A_envelope, ACH, volume):
    """
    Takes all 3 steps together
    """
    RH_internal = indoor_humidity(T_setpoint_F, T_external_F, RH_external)
    #print("Internal RH: ", RH_internal)
    Q_total = building_heat_loss(T_setpoint_F, T_external_F, R_wall, A_envelope, ACH, volume)
    #print("Building heat loss: ", Q_total)
    T_wall_F = wall_temp(T_setpoint_F, Q_total, A_envelope)
    #print("Wall temperature (F): ", T_wall_F)
    q_body, verdict = comfort_model(T_setpoint_F, RH_internal, T_wall_F)

    return Q_total, RH_internal, q_body, verdict

# Sweep outdoor conditions to build database

## Reference values for:
### ACH (Air Changes per Hour by infiltration): 0.5
### R_wall: 2.3 $m^2 K/W$
### R_film: 0.13 $m^2 K/W$
### Volume: 30 $m^3$ for a room (500 for whole home)
### Area: 70 $m^2$ for a room (500 for whole home)
We might make adjustments to these values later

In [60]:
# December 2025, row 28
s = steady_state_model(68, 32, 0.6599, 3, 70, 0.5, 30)
print()
print("Heat loss of user (watts): ", s[0])
print("Verdict: ", s[1])
#steady_state_model(68, 32, 0.6599, 2.3, 500, 0.5, 500)


Heat loss of user (watts):  68.71242046438249
Verdict:  Good


In [61]:
# December 2025, row 28
s = steady_state_model(72, 32, 0.6599, 3, 70, 0.5, 30)
print()
print("Heat loss of user (watts): ", s[0])
print("Verdict: ", s[1])


Heat loss of user (watts):  59.03931448977549
Verdict:  Good


In [62]:
# December 2025, row 28
s = steady_state_model(76, 32, 0.6599, 3, 70, 0.5, 30)
print()
print("Heat loss of user (watts): ", s[0])
print("Verdict: ", s[1])


Heat loss of user (watts):  49.22276341438757
Verdict:  Good


In [63]:
# Later we will test our model on weather data

In [64]:
import pandas as pd

df = pd.read_csv("Weather and Temperature Logs(February 2026).csv")

In [65]:
df.head()

,Time Stamp,Current Weather,Temp (C),Humidity,Barometric Pressue (Pa),Wind Direction,Wind Speed (m/s),Wind Chill (C),Heat Index (C)
0,2/1/2026 0:15:00,Cloudy,-11.4,60.644017,101420.0,320.0,14.76,-18.385102,NaN
1,2/1/2026 0:35:00,Mostly Cloudy,-11.5,61.647019,101390.0,310.0,9.36,-16.787109,NaN
2,2/1/2026 0:55:00,Clear,-11.6,61.620911,101320.0,310.0,22.32,-20.310614,NaN
3,2/1/2026 1:15:00,Clear,-12.1,64.140316,101360.0,320.0,16.56,-19.708570,NaN
4,2/1/2026 1:35:00,Clear,-12.3,64.090342,101390.0,310.0,12.96,-18.978616,NaN


In [66]:
df.dtypes

Time Stamp                  object
Current Weather             object
Temp (C)                   float64
Humidity                   float64
Barometric Pressue (Pa)    float64
Wind Direction             float64
Wind Speed (m/s)           float64
Wind Chill (C)             float64
Heat Index (C)             float64
dtype: object

In [67]:
# Convert outdoor temp to Fahrenheit
df["Temp_F"] = df["Temp (C)"] * 9/5 + 32
df["Humidity_fraction"] = df["Humidity"] / 100


# Run minimum setpoint search across each weather observation
T_setpoint_range = np.arange(65, 76, 1)  # °F

min_setpoints = []
for _, row in df.iterrows():
    T_ext = row["Temp_F"]
    RH_ext = row["Humidity_fraction"]
    
    min_set = None
    for T_set in T_setpoint_range:
        _, verdict = steady_state_model(T_set, T_ext, RH_ext, 
                                         R_wall=3, A_envelope=70, 
                                         ACH=0.5, volume=30)
        if verdict == "Good":
            min_set = T_set
            break  # lowest setpoint that achieves comfort
    
    min_setpoints.append(min_set)

df["min_setpoint_F"] = min_setpoints
print(df[["Time Stamp", "Temp (C)", "Humidity", "Temp_F", "min_setpoint_F"]].head(20))

          Time Stamp  Temp (C)   Humidity  Temp_F  min_setpoint_F
0   2/1/2026 0:15:00     -11.4  60.644017   11.48              65
1   2/1/2026 0:35:00     -11.5  61.647019   11.30              65
2   2/1/2026 0:55:00     -11.6  61.620911   11.12              65
3   2/1/2026 1:15:00     -12.1  64.140316   10.22              65
4   2/1/2026 1:35:00     -12.3  64.090342    9.86              65
5   2/1/2026 1:55:00     -12.4  64.608296    9.68              65
6   2/1/2026 2:15:00     -12.6  67.907452    9.32              65
7   2/1/2026 2:35:00     -12.6  69.639389    9.32              65
8   2/1/2026 2:55:00     -12.6  69.639389    9.32              65
9   2/1/2026 3:15:00     -12.7  71.389661    9.14              65
10  2/1/2026 3:35:00     -12.7  71.989354    9.14              65
11  2/1/2026 3:55:00     -12.6  72.010068    9.32              65
12  2/1/2026 4:15:00     -12.6  72.613901    9.32              65
13  2/1/2026 4:35:00     -12.7  76.314907    9.14              65
14  2/1/20

In [71]:
# Debug single row
T_ext = 11.48
RH_ext = 0.606

for T_set in T_setpoint_range:
    Q_loss, RH_int, q_body, verdict = steady_state_model(T_set, T_ext, RH_ext,
                                                          R_wall=3, A_envelope=70,
                                                          ACH=0.5, volume=30)
    print(f"T_set={T_set}, RH_int={RH_int:.3f}, q_body={q_body:.2f}, verdict={verdict}")

T_set=65, RH_int=0.074, q_body=78.09, verdict=Good
T_set=66, RH_int=0.071, q_body=75.70, verdict=Good
T_set=67, RH_int=0.069, q_body=73.31, verdict=Good
T_set=68, RH_int=0.066, q_body=70.90, verdict=Good
T_set=69, RH_int=0.064, q_body=68.49, verdict=Good
T_set=70, RH_int=0.062, q_body=66.06, verdict=Good
T_set=71, RH_int=0.060, q_body=63.63, verdict=Good
T_set=72, RH_int=0.058, q_body=61.20, verdict=Good
T_set=73, RH_int=0.056, q_body=58.75, verdict=Good
T_set=74, RH_int=0.054, q_body=56.29, verdict=Good
T_set=75, RH_int=0.052, q_body=53.83, verdict=Good


In [ ]:
for T_set in T_setpoint_range:
    Q_total = building_heat_loss(T_set, 11.48, 3, 70, 0.5, 30)
    T_wall = wall_temp(T_set, Q_total, 70)
    print(f"T_set={T_set}, Q_total={Q_total:.1f}, T_wall={T_wall:.2f}F")

T_set=65, Q_total=843.2, T_wall=62.18F
T_set=66, Q_total=858.9, T_wall=63.13F
T_set=67, Q_total=874.7, T_wall=64.08F
T_set=68, Q_total=890.5, T_wall=65.02F
T_set=69, Q_total=906.2, T_wall=65.97F
T_set=70, Q_total=922.0, T_wall=66.92F
T_set=71, Q_total=937.7, T_wall=67.87F
T_set=72, Q_total=953.5, T_wall=68.81F
T_set=73, Q_total=969.2, T_wall=69.76F
T_set=74, Q_total=985.0, T_wall=70.71F
T_set=75, Q_total=1000.7, T_wall=71.65F


: 